In [42]:
import pandas  as pd
import numpy as np
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from utils import(
    analyze_dataframe,
    plot_dataframe,
    plot_numerical_target,
    calculate_relationship
)

from stat_utils import statistical_tests_step1#, statistical_tests_step2

pd.set_option('display.max_columns',None)
#pd.set_option('displat.max_rows',100)

In [43]:
df=pd.read_csv('artifacts/train_data.csv')

target_columns = '잔액_리볼빙일시불이월_target'

In [44]:
analysis_columns = [col for col in df.columns if col != target_columns]

print(f'Total columns to analyze: {len(analysis_columns)}')
print(f'Target columns: {target_columns}')
print(f'Target unique values: {df[target_columns].nunique()}')

Total columns to analyze: 226
Target columns: 잔액_리볼빙일시불이월_target
Target unique values: 79209


In [45]:
df['연령'].unique()

array(['40대', '30대', '60대', '50대', '20대', '70대이상'], dtype=object)

In [46]:
print(df['연령'].value_counts().sort_index())
print(df['Life_Stage'].value_counts().sort_index())

연령
20대       8638
30대      25067
40대      27558
50대      16130
60대       5070
70대이상      940
Name: count, dtype: int64
Life_Stage
1.Single       4098
2.가족형성기       10137
3.자녀출산기       10445
4.자녀성장기(1)    31862
5.자녀성장기(2)    18600
6.자녀출가기        4692
7.노령           3569
Name: count, dtype: int64


In [47]:
category_cols =[
    '이용금액_쇼핑',
    '이용금액_요식',
    '이용금액_교통',
    '이용금액_의료',
    '이용금액_납부',
    '이용금액_교육',
    '이용금액_여유생활',
    '이용금액_사교활동',
    '이용금액_일상생활',
    '이용금액_해외'
]
category_by_age = df.groupby('연령')[category_cols].mean().round(0)

df['총_업종별_사용금액']=df[category_cols].sum(axis=1)

for col in category_cols:
    ratio_col = col.replace('이용금액_','')+'_비율'
    df[ratio_col]=np.where(
        df['총_업종별_사용금액'] >0,
        df[col] / df['총_업종별_사용금액'] * 100,
        0
    )

ratio_cols = [col.replace('이용금액_','')+'_비율' for col in category_cols]
ratio_by_age = df.groupby('연령')[ratio_cols].mean().round(2)
print(ratio_by_age)

       쇼핑_비율  요식_비율  교통_비율  의료_비율  납부_비율  교육_비율  여유생활_비율  사교활동_비율  일상생활_비율  \
연령                                                                           
20대    43.07   1.95  10.50   1.55  11.66   0.46     1.32    16.70     1.01   
30대    41.44   1.35  11.11   1.82  14.57   1.71     0.98    13.92     0.85   
40대    37.79   0.96  12.27   1.99  15.87   2.72     1.11    14.26     0.90   
50대    34.50   0.66  14.84   2.28  16.70   0.65     1.13    13.89     0.85   
60대    34.21   0.43  15.15   2.95  16.25   0.24     0.88    12.92     0.75   
70대이상  36.43   0.56  12.33   4.91   8.54   0.49     0.74    14.96     0.61   

       해외_비율  
연령            
20대     4.11  
30대     3.74  
40대     2.75  
50대     2.69  
60대     2.34  
70대이상   1.82  


# 연령 별 카테고리_비율도 활용하고 싶었는데

# statistical_tests_step2가 stat_utils에 없어서 확인 작업만 하고 끝냈습니다.

------------------------------------------------------------------------------------------------------------------------------------------------

# 1. Base Model 선정하기 

###  EBM(Explainable Boosting Regressor)

타겟 변수인 '잔액_리볼빙일시불이월_target'은 '잔액 규모', '카드 이용 패턴', 'RV 관련 지표' 등으로 인한 '비선형적 특성'을 가질 확률이 높다.

해당 모델이 비선형 관계를 학습하면서도 각 변수의 기여도를 개별적으로 해석 가능하다는 점과

변수 중요도 순위를 제시한다는 점, Feature selection 결과를 정량적으로 비교, 검증이 가능하다는 점에서 적절하다고 판단했다.


# 2. Base Model에 들어가는 변수 선택하기

### - Method 0 : 도메인 지식 기반 수작업 변수 선정

이전 과제에서의 EDA 및 가설 검증 결과를 바탕으로 타겟 변수와 직접적인 관계가 있다고 판단되는 변수들을 선정했다.

: 

- '잔액_리볼빙일시불이월_B0M' : 타겟 변수의 직전월 데이터

- 'RV약정청구율' : 리볼빙 잔액의 증가/감소와 직접적인 관계 O

- '카드이용한도금액', '잔액_신판평균한도소진율_r6m' : 고객의 한도와 관련한 변수가 RV이용으로 이어질 수 있다고 판단

- '이용금액_일시불_B0M' : 일시불 이용금액이 클수록 RV를 이용할 가능성이 높다고 판단

- '이용금액_신용_B0M' : 위와 비슷한 이유로 신용 이용금액이 클수록 RV를 이용할 가능성이 높다고 판단

- '이용건수_신용_B0M' : 이용건수가 높을수록 이용금액이 커지고, 이것이 RV 이용으로 이어진다고 판단



###  - Method 1 : 카테고리별로 작은 모델 만들기 -> 실패

### - Method 2 : 통계적 유의성 기반 Feature Selection (EDA 결과 활용) _ 강사님 코드에서 추출된 변수들

- 연속형 변수: Pearson / Spearman 상관계수
- 범주형 변수: Kruskal-Wallis 검정
- 기준: p-value < 0.05 또는 |corr| ≥ threshold

그 결과, 아래 변수들이 타겟 변수(잔액_리볼빙일시불이월_target)과
통계적으로 유의한 관계를 가지는 것으로 확인되어
Base Model의 입력 변수로 추가하였다.


또한 중복되는 변수 간의 정보 중복을 줄이기 위해

잔액_일시불_BOM/B1M/B2M은 평균값으로 통합하여 ‘잔액_일시불_평균_B012M’ 파생변수를 생성하였다.


In [48]:
# 잔액_일시불_B0M/B1M/B2M 평균 파생변수
inst_cols = ['잔액_일시불_B0M', '잔액_일시불_B1M', '잔액_일시불_B2M']


missing = [c for c in inst_cols if c not in df.columns]
if missing:
    raise KeyError(f"다음 컬럼이 df에 없습니다: {missing}")

df['잔액_일시불_평균_B012M'] = df[inst_cols].mean(axis=1)

# 원래 3개 컬럼 제거
df = df.drop(columns=inst_cols)


final_cols = [

    # 도메인 기반 선정 컬럼
    '카드이용한도금액',
    '이용금액_일시불_B0M',
    '이용금액_신용_B0M',
    '이용건수_신용_B0M',
    '잔액_신판평균한도소진율_r6m',


    '연령',
    'VIP등급코드',
    '최상위카드등급코드',
    '거주시도명',
    '직장시도명',
    'Life_Stage',
    'RV약정청구율',
    'RV전환가능여부',
    '잔액_B0M',
    '잔액_일시불_평균_B012M',          # <- 새로 만든 평균 컬럼
    '잔액_리볼빙일시불이월_B0M',
    '월중평잔_일시불_B0M',
    'RV_평균잔액_R12M',
    'RV_최대잔액_R12M',
    '월중평잔_일시불',
    '월중평잔_RV일시불',
    '평잔_일시불_3M',
    '평잔_RV일시불_3M'


]

# 실제 df에 없는 컬럼 있으면 바로 확인
missing2 = [c for c in final_cols if c not in df.columns]
if missing2:
    raise KeyError(f"final_cols 중 df에 없는 컬럼: {missing2}")

df_model = df[final_cols].copy()

print(df_model.shape)
df_model.head()


(83403, 23)


,카드이용한도금액,이용금액_일시불_B0M,이용금액_신용_B0M,이용건수_신용_B0M,잔액_신판평균한도소진율_r6m,연령,VIP등급코드,최상위카드등급코드,거주시도명,직장시도명,Life_Stage,RV약정청구율,RV전환가능여부,잔액_B0M,잔액_일시불_평균_B012M,잔액_리볼빙일시불이월_B0M,월중평잔_일시불_B0M,RV_평균잔액_R12M,RV_최대잔액_R12M,월중평잔_일시불,월중평잔_RV일시불,평잔_일시불_3M,평잔_RV일시불_3M
0,4209755,530723,1371569,7,0.469525,40대,07,_,부산,부산,4.자녀성장기(1),20.912946,N,6148994,2.619313e+06,1512611,2541927,2249140,2964268,2431626,3709992,2245452,4062640
1,387324,0,0,0,0.847372,30대,06,_,대구,대구,4.자녀성장기(1),20.492362,N,2378072,1.406054e+06,1139341,1273831,1529022,2175153,1117335,1481292,1190321,1464088
2,8638043,134267,848926,9,0.746399,60대,07,_,충남,충남,6.자녀출가기,19.625443,N,10621291,3.319367e+06,1936044,2887384,3803153,3772276,2735060,3292995,2736418,3947971
3,8792092,336233,1092614,10,0.022702,40대,07,_,전북,전북,4.자녀성장기(1),100.000000,Z,10380384,9.141817e+05,894886,1302398,317252,890150,1158498,1621869,1949992,2039534
4,5140724,705905,1823372,31,0.272367,50대,07,_,인천,인천,5.자녀성장기(2),22.855123,N,4914003,1.611284e+06,1024403,2675803,831746,1541686,2077043,1470200,2087627,1535652


In [49]:
target_col = '잔액_리볼빙일시불이월_target'  

y = df[target_col]
X = df_model

cat_cols = ['VIP등급코드','최상위카드등급코드','거주시도명','직장시도명','Life_Stage','RV전환가능여부']

X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)


In [50]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_reg(y_true, y_pred, name="model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))   # 버전 호환
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}] MAE={mae:,.0f}  RMSE={rmse:,.0f}  R2={r2:.4f}")


target_col = "잔액_리볼빙일시불이월_target"
y = df[target_col].copy()
X = df_model.copy()    # 너가 만든 final_cols로 만든 df_model

# y 숫자화
y = pd.to_numeric(y, errors="coerce")

# 범주형 자동 더미
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# 혹시 남은 object 제거 + 결측 처리
X_enc = X_enc.apply(pd.to_numeric, errors="coerce").fillna(0)

# y 결측 제거
mask = y.notna()
X_enc = X_enc.loc[mask].copy()
y = y.loc[mask].copy()

print("X_enc:", X_enc.shape, "y:", y.shape)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42
)


X_enc: (83403, 68) y: (83403,)


In [52]:
from interpret.glassbox import ExplainableBoostingRegressor

ebm = ExplainableBoostingRegressor(random_state=42)
ebm.fit(X_train, y_train)

pred_ebm = ebm.predict(X_test)
pred_ebm = np.clip(pred_ebm, 0, None)
eval_reg(y_test, pred_ebm, "EBM (raw y)")

gi = ebm.explain_global()
names = gi.data()["names"]
scores = gi.data()["scores"]

imp = pd.DataFrame({"feature": names, "importance": scores}).sort_values("importance", ascending=False)
display(imp.head(30))


[EBM (raw y)] MAE=160,719  RMSE=216,335  R2=0.7772


,feature,importance
10,RV_평균잔액_R12M,139919.260185
8,잔액_리볼빙일시불이월_B0M,72418.662885
13,월중평잔_RV일시불,46168.920429
7,잔액_일시불_평균_B012M,43706.854823
14,평잔_일시불_3M,40068.234641
15,평잔_RV일시불_3M,35523.940673
4,잔액_신판평균한도소진율_r6m,29989.520864
5,RV약정청구율,22802.374814
0,카드이용한도금액,22379.141159
9,월중평잔_일시불_B0M,17919.957664


### - Method 3 : Boruta를 통한 유의미한 변수 남기기

### Method0와 Method2에서 선별한 변수들(final_cols)를 가지고 진행

In [53]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from boruta import BorutaPy


# target_col = "잔액_리볼빙일시불이월_target"


# 1) X, y 만들기 
X = df[final_cols].copy()
y = pd.to_numeric(df[target_col], errors="coerce")

# y 결측 제거
mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].copy()

# 2) 범주형 원-핫 
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=False, prefix_sep="=")

# 3) 숫자 변환 + 결측 처리
X_enc = X_enc.apply(pd.to_numeric, errors="coerce").fillna(0)

# 4) train만으로 Boruta(데이터 누수 방지)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_enc, y, test_size=0.2, random_state=42
)

# 5) Boruta용 모델
rf = RandomForestRegressor(
    n_estimators=800,
    random_state=42,
    n_jobs=-1,
    max_depth=None,
    min_samples_leaf=5
)

boruta = BorutaPy(
    estimator=rf,
    n_estimators="auto",
    perc=100,
    alpha=0.05,
    two_step=True,
    random_state=42,
    verbose=0
)

boruta.fit(X_train.values, y_train.values)

# 6) 더미(feature) 단위 결과
feat_names = np.array(X_train.columns)
selected_mask = boruta.support_
ranking = boruta.ranking_

result_df = pd.DataFrame({
    "feature": feat_names,
    "selected": selected_mask,
    "ranking": ranking
}).sort_values(["selected", "ranking"], ascending=[False, True])

print("✅ Selected dummy-features:", selected_mask.sum(), "/", len(selected_mask))
display(result_df.head(30))

# 7) 원래 컬럼 단위로 묶기 
def to_original_col(feat: str) -> str:
    return feat.split("=", 1)[0]  # '거주시도명=경기' -> '거주시도명'
                                

result_df["orig_col"] = result_df["feature"].apply(to_original_col)

# 원래 컬럼이 하나라도 selected dummy를 가지면 "선택"으로 판단
orig_summary = (result_df
    .groupby("orig_col")["selected"]
    .any()
    .reset_index()
    .rename(columns={"selected": "selected_any_dummy"})
    .sort_values("selected_any_dummy", ascending=False)
)

selected_orig = orig_summary.loc[orig_summary["selected_any_dummy"], "orig_col"].tolist()
rejected_orig = orig_summary.loc[~orig_summary["selected_any_dummy"], "orig_col"].tolist()

print("\n✅ Boruta가 '살린' 원래 변수 수:", len(selected_orig))
print("✅ Boruta가 '버린' 원래 변수 수:", len(rejected_orig))

print("\n[살아남은 변수들]")
print(selected_orig)

print("\n[탈락 변수들]")
print(rejected_orig)

display(orig_summary)


✅ Selected dummy-features: 10 / 75


,feature,selected,ranking
0,카드이용한도금액,True,1
2,이용금액_신용_B0M,True,1
4,잔액_신판평균한도소진율_r6m,True,1
6,잔액_B0M,True,1
7,잔액_일시불_평균_B012M,True,1
8,잔액_리볼빙일시불이월_B0M,True,1
10,RV_평균잔액_R12M,True,1
11,RV_최대잔액_R12M,True,1
13,월중평잔_RV일시불,True,1
15,평잔_RV일시불_3M,True,1



✅ Boruta가 '살린' 원래 변수 수: 10
✅ Boruta가 '버린' 원래 변수 수: 13

[살아남은 변수들]
['잔액_B0M', 'RV_평균잔액_R12M', '평잔_RV일시불_3M', '카드이용한도금액', '잔액_일시불_평균_B012M', '월중평잔_RV일시불', '잔액_신판평균한도소진율_r6m', '잔액_리볼빙일시불이월_B0M', 'RV_최대잔액_R12M', '이용금액_신용_B0M']

[탈락 변수들]
['Life_Stage', '최상위카드등급코드', '직장시도명', '이용건수_신용_B0M', '이용금액_일시불_B0M', '월중평잔_일시불_B0M', '월중평잔_일시불', '연령', '거주시도명', 'VIP등급코드', 'RV전환가능여부', 'RV약정청구율', '평잔_일시불_3M']


,orig_col,selected_any_dummy
14,잔액_B0M,True
2,RV_평균잔액_R12M,True
21,평잔_RV일시불_3M,True
20,카드이용한도금액,True
17,잔액_일시불_평균_B012M,True
8,월중평잔_RV일시불,True
16,잔액_신판평균한도소진율_r6m,True
15,잔액_리볼빙일시불이월_B0M,True
1,RV_최대잔액_R12M,True
12,이용금액_신용_B0M,True


# 3. Base Model 실행하여 결과 측정

In [54]:
# Boruta를 통해 선별된 변수들

selected_cols = [
    '잔액_B0M',
    'RV_평균잔액_R12M',
    '평잔_RV일시불_3M',
    '카드이용한도금액',
    '잔액_일시불_평균_B012M',
    '월중평잔_RV일시불',
    '잔액_신판평균한도소진율_r6m',
    '잔액_리볼빙일시불이월_B0M',
    'RV_최대잔액_R12M',
    '이용금액_신용_B0M'
]

target = '잔액_리볼빙일시불이월_target'


### 학습용 데이터 구성

In [55]:
# 타겟 + 선택 변수만 사용
df_final = df[selected_cols + [target]].dropna()

X = df_final[selected_cols]
y = df_final[target]

# 범주형 자동 인식 → 더미 처리
X = pd.get_dummies(X, drop_first=True)


In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### EBM 학습

In [57]:
ebm = ExplainableBoostingRegressor(
    random_state=42
)

ebm.fit(X_train, y_train)

ExplainableBoostingRegressor()

### 성능 평가 (Base Model 결과)

In [64]:
pred = ebm.predict(X_test)
pred = np.clip(pred, 0, None)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(y_test, pred)   
rmse = np.sqrt(mse)                      

print(f"EBM (Boruta-selected features)")
print(f'MAE  : {mae:,.0f}', end=' / ')
print(f"RMSE : {rmse:,.0f}", end=' / ')
print(f"R²   : {r2:.4f}")
print(f"사용 변수 수: {X.shape[1]}")


EBM (Boruta-selected features)
MAE  : 161,632 / RMSE : 217,902 / R²   : 0.7739
사용 변수 수: 10


In [59]:
gi = ebm.explain_global()

importance_df = pd.DataFrame({
    "feature": gi.data()["names"],
    "importance": gi.data()["scores"]
}).sort_values("importance", ascending=False)

importance_df.head(10)

,feature,importance
1,RV_평균잔액_R12M,169443.972116
7,잔액_리볼빙일시불이월_B0M,75976.608656
5,월중평잔_RV일시불,43762.849345
2,평잔_RV일시불_3M,25680.842686
9,이용금액_신용_B0M,25348.641763
6,잔액_신판평균한도소진율_r6m,23497.859252
4,잔액_일시불_평균_B012M,21880.592289
3,카드이용한도금액,21437.047545
8,RV_최대잔액_R12M,15450.116331
0,잔액_B0M,5168.613308


### boruta 이전 : MAE=160,719  RMSE=216,335  R2=0.7772

### boruta 이후 : MAE  : 161,632  RMSE : 217,902  R²   : 0.7739